# Auto Schema Generation

In [2]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()

DATABRICKS_TOKEN = os.environ.get("DATABRICKS_TOKEN")
DATABRICKS_HOST = os.environ.get("DATABRICKS_HOST")
DATABRICKS_ENDPOINT = os.environ.get("DATABRICKS_ENDPOINT")

client = OpenAI(
    api_key=DATABRICKS_TOKEN,
    base_url=f"{DATABRICKS_HOST}/serving-endpoints",
)

We will auto-generate the OpenAI tool schema directly from this function's type hints and docstring.

In [3]:
def get_weather(location: str, unit: str = "celsius"):
    """Get current weather for a city.

    Args:
        location: City name, e.g. Seattle
        unit: celsius or fahrenheit
    """
    sample = {
        "seattle": {"celsius": 16, "fahrenheit": 61, "condition": "Cloudy"},
        "austin": {"celsius": 31, "fahrenheit": 88, "condition": "Hot"},
        "new york": {"celsius": 22, "fahrenheit": 72, "condition": "Sunny"},
    }
    weather = sample.get(location.strip().lower(), {"celsius": 24, "fahrenheit": 75, "condition": "Clear"})
    return {
        "location": location,
        "unit": unit,
        "temperature": weather[unit],
        "condition": weather["condition"],
    }

## 2) Auto-Build the Tool Schema from Signature

This helper inspects function parameters and creates `tools=[...]` automatically.

In [4]:
import inspect
import json
from typing import get_origin, get_args


def _python_type_to_json_schema(annotation):
    if annotation is inspect._empty:
        return {"type": "string"}

    origin = get_origin(annotation)
    args = get_args(annotation)

    if annotation is str:
        return {"type": "string"}
    if annotation is int:
        return {"type": "integer"}
    if annotation is float:
        return {"type": "number"}
    if annotation is bool:
        return {"type": "boolean"}

    if origin is list and args:
        return {"type": "array", "items": _python_type_to_json_schema(args[0])}

    if origin is dict:
        return {"type": "object"}

    return {"type": "string"}


def build_tool_from_function(func):
    sig = inspect.signature(func)
    properties = {}
    required = []

    for name, param in sig.parameters.items():
        schema = _python_type_to_json_schema(param.annotation)

        if name == "unit":
            schema["enum"] = ["celsius", "fahrenheit"]

        properties[name] = schema

        if param.default is inspect._empty:
            required.append(name)

    return {
        "type": "function",
        "function": {
            "name": func.__name__,
            "description": (func.__doc__ or "").strip() or f"Call {func.__name__}",
            "parameters": {
                "type": "object",
                "properties": properties,
                "required": required,
            },
        },
    }


weather_tool = build_tool_from_function(get_weather)
tools = [weather_tool]

print(json.dumps(weather_tool, indent=2))

{
  "type": "function",
  "function": {
    "name": "get_weather",
    "description": "Get current weather for a city.\n\nArgs:\n    location: City name, e.g. Seattle\n    unit: celsius or fahrenheit",
    "parameters": {
      "type": "object",
      "properties": {
        "location": {
          "type": "string"
        },
        "unit": {
          "type": "string",
          "enum": [
            "celsius",
            "fahrenheit"
          ]
        }
      },
      "required": [
        "location"
      ]
    }
  }
}


## 3) Use the Auto-Generated Tool in a Real Call

In [5]:
messages = [
    {"role": "system", "content": "You are helpful. Use tools when needed."},
    {"role": "user", "content": "What is the weather in Seattle in fahrenheit?"},
]

first = client.chat.completions.create(
    model=DATABRICKS_ENDPOINT,
    messages=messages,
    tools=tools,
    tool_choice="auto",
    temperature=0,
)

assistant_msg = first.choices[0].message
messages.append(
    {
        "role": "assistant",
        "content": assistant_msg.content or "",
        "tool_calls": [tc.model_dump() for tc in (assistant_msg.tool_calls or [])],
    }
)

if assistant_msg.tool_calls:
    tc = assistant_msg.tool_calls[0]
    args = json.loads(tc.function.arguments)

    tool_result = get_weather(**args)

    messages.append(
        {
            "role": "tool",
            "tool_call_id": tc.id,
            "content": json.dumps(tool_result),
        }
    )

    final = client.chat.completions.create(
        model=DATABRICKS_ENDPOINT,
        messages=messages,
        tools=tools,
        temperature=0,
    )

    print("Tool picked by model:", tc.function.name)
    print("Tool args:", args)
    print("Tool result:", tool_result)
    print("Final answer:", final.choices[0].message.content)
else:
    print("No tool call returned. Re-run this cell.")

Tool picked by model: get_weather
Tool args: {'location': 'Seattle', 'unit': 'fahrenheit'}
Tool result: {'location': 'Seattle', 'unit': 'fahrenheit', 'temperature': 61, 'condition': 'Cloudy'}
Final answer: The current weather in Seattle is 61 degrees Fahrenheit and it is cloudy.


## 4) Reuse the Same Builder for Another Function

Here we define `add_numbers` and generate its schema with the same `build_tool_from_function(...)` helper.

In [6]:
def add_numbers(a: int, b: int):
    """Add two integers and return the sum."""
    return {"sum": a + b}


add_tool = build_tool_from_function(add_numbers)
print("Auto-generated schema for add_numbers:\n")
print(json.dumps(add_tool, indent=2))

messages_add = [
    {"role": "user", "content": "What is 17 + 25? Use the add_numbers tool."},
]

first_add = client.chat.completions.create(
    model=DATABRICKS_ENDPOINT,
    messages=messages_add,
    tools=[add_tool],
    tool_choice="auto",
    temperature=0,
)

assistant_add = first_add.choices[0].message
messages_add.append(
    {
        "role": "assistant",
        "content": assistant_add.content or "",
        "tool_calls": [tc.model_dump() for tc in (assistant_add.tool_calls or [])],
    }
)

if assistant_add.tool_calls:
    tc = assistant_add.tool_calls[0]
    args = json.loads(tc.function.arguments)
    tool_result = add_numbers(**args)

    messages_add.append(
        {
            "role": "tool",
            "tool_call_id": tc.id,
            "content": json.dumps(tool_result),
        }
    )

    final_add = client.chat.completions.create(
        model=DATABRICKS_ENDPOINT,
        messages=messages_add,
        tools=[add_tool],
        temperature=0,
    )

    print("\nTool picked:", tc.function.name)
    print("Tool args:", args)
    print("Tool result:", tool_result)
    print("Final answer:", final_add.choices[0].message.content)
else:
    print("No tool call returned. Re-run this cell.")

Auto-generated schema for add_numbers:

{
  "type": "function",
  "function": {
    "name": "add_numbers",
    "description": "Add two integers and return the sum.",
    "parameters": {
      "type": "object",
      "properties": {
        "a": {
          "type": "integer"
        },
        "b": {
          "type": "integer"
        }
      },
      "required": [
        "a",
        "b"
      ]
    }
  }
}

Tool picked: add_numbers
Tool args: {'a': 17, 'b': 25}
Tool result: {'sum': 42}
Final answer: The answer is 42.
